<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day01_practice3_%EC%B2%AB%EB%94%A5%EB%9F%AC%EB%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 첫 딥러닝 실습 - sklearn과 나란히 비교

import torch
import torch.nn as nn #Neural Network(신경망) 모듈
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

torch.manual_seed(42)

device = "cuda" if torch.cuda. is_available() else "cpu"
print(device)

cpu


In [8]:
# 셀 1. 데이터 준비 - 여기까지는 머신러닝과 동일
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler() # 표준화란 각 특성(feature)의 평균을 0으로, 표준 편차를 1로 조정
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")
print(f"y 데이터: {y_train.shape}, y 테스트: {y_test.shape}")

학습 데이터: (455, 30), 테스트 데이터: (114, 30)
y 데이터: (455,), y 테스트: (114,)


In [7]:
# 셀 2. 머신러닝 방식 - sklearn
ml_model = LogisticRegression(max_iter=1000) # 이진 분류, 유방암의 악성/양성을 예측
ml_model.fit(X_train, y_train)
ml_acc = accuracy_score(y_test, ml_model.predict(X_test))
print(f"sklearn 로지스틱 회귀 머신러닝 모델 정확도: {ml_acc:.4f}")

sklearn 로지스틱 회귀 머신러닝 모델 정확도: 0.9737


In [10]:
# 셀 3. 딥러닝 방식 - 데이터를 텐서로
# 넘파이 → 텐서, 그리고 device로
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1).to(device) # (455,) 1차원 벡터 → (455, 1) 2차원 형태로 변환
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1,1).to(device)

print("텐서 변환:", X_train_t.shape, y_train_t.shape)

텐서 변환: torch.Size([455, 30]) torch.Size([455, 1])


In [13]:
# 셀 4. 신경망 정의
model = nn.Sequential(
    nn.Linear(30, 16),
    nn.ReLU(),
    nn.Linear(16,1),
    nn.Sigmoid()
).to(device)

print("모델 구조:")
print(model)

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

모델 구조:
Sequential(
  (0): Linear(in_features=30, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=1, bias=True)
  (3): Sigmoid()
)


In [15]:
# 셀 5. 학습 루프
for epoch in range(100):
  pred = model(X_train_t)
  loss = loss_fn(pred, y_train_t)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if(epoch + 1) % 20 == 0:
    print(f"Epoch: {epoch + 1:3d}, Loss: {loss.item():.4f}")

Epoch:  20, Loss: 0.0314
Epoch:  40, Loss: 0.0257
Epoch:  60, Loss: 0.0206
Epoch:  80, Loss: 0.0163
Epoch: 100, Loss: 0.0130


In [19]:
# 셀 6. 평가 - 두 방식 비교
model.eval()
with torch.no_grad():
  test_pred = model(X_test_t)
  test_acc = ((test_pred > 0.5) == y_test_t.bool()).float().mean().item()

print(f" [sklearn 로지스틱 회귀] 정확도: {ml_acc:.4f}")
print(f" [PyTorch 신경망] 정확도: {test_acc:.4f}")

 [sklearn 로지스틱 회귀] 정확도: 0.9737
 [PyTorch 신경망] 정확도: 0.9825
